# 专题: GUI架构解析

**相关文件:**
- `gui/main.py`
- `gui/web/app.js`
- `gui/web/index.html`

## 目的
本项目包含一个图形用户界面（GUI），用于实现模型的快速、可视化建模。此文档旨在解析GUI的技术架构，为希望理解、修改或扩展GUI功能的开发者提供指导。

**注意:** 本文档只解析架构，并不能实际运行GUI。

此笔记本将：
1.  介绍GUI的核心技术选型：`Eel`库。
2.  剖析Python后端（`main.py`）的职责。
3.  剖析JavaScript前端（`app.js`）的职责。
4.  展示前后端之间如何进行双向通信。

## 1. 技术架构: Eel

GUI采用[Eel](https://github.com/python-eel/Eel)库构建。Eel是一个轻量级的Python库，用于制作简单的、基于Web技术的桌面应用程序。其核心思想是：

- **后端**: 使用Python来处理所有的数据计算和文件操作。在这个项目中，就是指运行水文水力模型的仿真计算。
- **前端**: 使用HTML, CSS, 和 JavaScript来构建用户界面。这允许开发者利用成熟的Web技术来创建丰富的交互体验。
- **通信**: Eel在Python后端和JavaScript前端之间建立一个WebSocket连接，允许它们像调用本地函数一样互相调用对方的函数。

这种架构的优点是兼顾了Python强大的计算能力和Web技术在界面开发上的灵活性。

## 2. Python后端 (`main.py`)

`main.py`是GUI应用的入口点和后端服务器。它的主要职责是：
- 初始化Eel，并指定前端文件所在的目录（`web`）。
- 定义一些函数，并使用`@eel.expose`装饰器将它们**暴露**给JavaScript前端，使其可以被前端调用。
- 启动Eel应用服务器。

### 关键代码片段: 暴露Python函数
下面这个函数`start_simulation`被暴露给前端。当用户在界面上点击“运行”按钮时，前端的JavaScript代码就会调用这个Python函数。

In [ ]:
# In gui/main.py

import eel

# ...

@eel.expose  # <--- 这个装饰器是关键
def start_simulation(config_path):
    """
    Loads a config file and runs the simulation, streaming updates to the front-end.
    """
    # ... (省略了内部实现) ...
    print(f"Python: Simulation started with {config_path}")
    # 在一个独立的线程中运行耗时的模拟
    # 并在循环中通过调用JS函数来更新前端状态
    # for status in controller.run(...):
    #     eel.update_status(status) # <--- 调用JS函数
    return "Simulation started."

## 3. JavaScript前端 (`app.js`)

`app.js`是前端界面的主要逻辑文件。它的职责是：
- 处理所有的用户交互，如拖拽组件、连接节点、修改参数等。
- 在需要执行计算时，调用后端暴露的Python函数。
- 定义一些函数，并使用`eel.expose`将它们**暴露**给Python后端，用于接收来自后端的更新。

### 关键代码片段: 调用Python函数
当用户点击运行按钮时，`handleRun`函数会被触发。在这个函数内部，通过`eel.start_simulation(...)`来调用Python后端的同名函数。

In [ ]:
// In gui/web/app.js

function handleRun() {
    // ... (清空日志等UI操作) ...
    console.log("JavaScript: Calling Python's start_simulation function.");
    
    // 调用Python函数，它返回一个JavaScript Promise
    eel.start_simulation('examples/config_coupled.yaml')().then(response => {
        console.log(`JavaScript: Received response from Python: ${response}`);
    });
}

### 关键代码片段: 暴露JavaScript函数
前端同样可以暴露函数给后端。例如，`update_status`函数被暴露出去。在Python后端进行长时间的模拟时，可以在每个时间步完成时，调用这个JS函数，将最新的状态（如流量、计算步数）发送给前端，从而实现“实时”的进度更新和绘图。

In [ ]:
// In gui/web/app.js

eel.expose(update_status, 'update_status'); // <--- 暴露JS函数
function update_status(status) {
    // ... (更新日志、图表等UI元素) ...
    console.log(`JavaScript: Received status update from Python for step ${status.step}`);
}

## 4. 总结

通过Eel库，本项目实现了一个经典的前后端分离的GUI架构：

1.  **用户操作**: 用户在浏览器前端（`index.html` + `app.js`）进行操作。
2.  **前端 -> 后端**: 当需要计算时，`app.js`调用`main.py`中暴露的Python函数（如`start_simulation`）。
3.  **后端处理**: `main.py`接收到请求，运行复杂的模型计算。
4.  **后端 -> 前端**: 在计算过程中或计算结束后，`main.py`可以随时调用`app.js`中暴露的JavaScript函数（如`update_status`），将数据和状态回传给前端。
5.  **前端渲染**: `app.js`接收到来自后端的数据，并更新界面上的图表、日志等元素，将结果呈现给用户。

这种松耦合的架构使得前后端的开发可以独立进行，是现代桌面应用开发的一种高效模式。